In [1]:
pip install -U langgraph langchain langchain-core langchain-community langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.6
    Uninstalling

In [2]:
# =========================
# IMPORTS
# =========================
from typing import TypedDict
from langgraph.graph import StateGraph, END

from langchain_groq import ChatGroq
import os


# =========================
# API KEY
# =========================
os.environ["GROQ_API_KEY"] = "******************"


# =========================
# DEFINE STATE
# =========================
class AgentState(TypedDict):
    input: str
    decision: str
    output: str


# =========================
# LLM INITIALIZATION
# =========================
llm = ChatGroq(model="llama-3.1-8b-instant")


# =========================
# NODE 1: INPUT PROCESSING
# =========================
def process_input(state: AgentState):
    user_input = state["input"]
    return {"input": user_input}


# =========================
# NODE 2: DECISION NODE (ROUTER)
# =========================
def decide(state: AgentState):
    text = state["input"].lower()

    if "search" in text:
        return {"decision": "search"}
    else:
        return {"decision": "llm"}


# =========================
# NODE 3: LLM RESPONSE
# =========================
def llm_node(state: AgentState):
    prompt = f"Answer this question: {state['input']}"
    response = llm.invoke(prompt)
    return {"output": response.content}


# =========================
# NODE 4: TOOL NODE (SEARCH SIMULATION)
# =========================
def search_node(state: AgentState):
    query = state["input"]
    result = f"Simulated search result for: {query}"
    return {"output": result}


# =========================
# BUILD GRAPH
# =========================
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("process", process_input)
graph.add_node("decide", decide)
graph.add_node("llm", llm_node)
graph.add_node("search", search_node)

# Entry point
graph.set_entry_point("process")

# Edges
graph.add_edge("process", "decide")

# Conditional routing
def route(state: AgentState):
    return state["decision"]

graph.add_conditional_edges(
    "decide",
    route,
    {
        "llm": "llm",
        "search": "search"
    }
)

# End connections
graph.add_edge("llm", END)
graph.add_edge("search", END)

# Compile graph
app = graph.compile()


# =========================
# RUN AGENT
# =========================
while True:
    user_input = input("\nEnter your query (type 'exit' to stop): ")

    if user_input.lower() == "exit":
        break

    result = app.invoke({"input": user_input})

    print("\nFinal Output:", result["output"])


Enter your query (type 'exit' to stop): search latest AI trends

Final Output: Simulated search result for: search latest AI trends

Enter your query (type 'exit' to stop): “search about neural networks”

Final Output: Simulated search result for: “search about neural networks”

Enter your query (type 'exit' to stop): “search machine learning applications”

Final Output: Simulated search result for: “search machine learning applications”

Enter your query (type 'exit' to stop): exit
